# Word Search

In [1]:
from typing import List

class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        # How I'd solve it, Scan through the array and check if there are letters within the word within the board
        # If there are letters within, usually I'd scan neighbors for all candidates and if exists scan again,
        # however I believe I can speed it up like an island union find, because it would be easier if we didn't double count the same exact (candidate word). it's ok to multi count 
        # letters to be part of different candidates.
        # where union if and only if word[i -1] or word[i +1] in the surrounding top, left. We don't have to do bottom right by iteration order invariant
        # we maintain an invariant that iterating left to right top to bottom, that on the scan of the matrix, 
        # in the set of all seen blocks the letter is put into all possible substring candidate word sets.
        candidates = dict()
        for i in range(len(board)):
            for j in range(len(board[0])):
                if i > 0: # there's top
                    board[i-1][j]
                if j > 0: # there's left
                    board[i][j-1]


                board[i][j]



In [4]:
def test(solution):
    cases = [
        (([["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]], "ABCCED"), True),
        (([["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]], "SEE"), True),
        (([["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]], "ABCB"), False),
        (([["A"]], "A"), True),
        (([["A"]], "B"), False),
        (([["A", "B"], ["C", "D"]], "ABCD"), False),
        (([["C", "A", "A"], ["A", "A", "A"], ["B", "C", "D"]], "AAB"), True),
    ]
    for i, (args, expected) in enumerate(cases, 1):
        got = solution(*args)
        assert got == expected, f'case {i}: expected {expected}, got {got}'


In [ ]:
from typing import List, OrderedDict
import copy

# from previously I'd normally do union find, however i also have to check the ordering and count and checking multiple adjacent hence there has to be a tree, this seems like backtracking.
class Helper:
    def __init__(self,board, word):
        self.board = board
        self.word = word

    def check_candidate_letter(self, k, l, i, j, seen):
        # Only allow moving to (k, l) if not seen
        if (k, l) in seen:
            return False
        # k, l = candidate position; i, j = from position
        letter = self.board[i][j]
        new_letter = self.board[k][l]
        for idx, c in enumerate(self.word):
            if c == letter:
                # check for adjacency in word
                if idx - 1 >= 0 and self.word[idx - 1] == new_letter:
                    return True
                if idx + 1 < len(self.word) and self.word[idx + 1] == new_letter:
                    return True
        return False

    def check_neighbors(self, i, j, word, path, seen):
        # print(f"path: {path}, self.board[{i}][{j}]: {self.board[i][j]}, seen: {seen}")

        board = self.board
        if "".join(path) == word:
            return True 
        seen.add((i, j))  # insert seen mark here

        # check top neighbor
        if i > 0 and self.check_candidate_letter(i - 1, j, i, j, seen):
            path.append(board[i-1][j])
            if self.check_neighbors(i - 1, j, word, path, seen):
                return True
            path.pop(-1)
        # check left neighbor
        if j > 0 and self.check_candidate_letter(i, j - 1, i, j, seen):
            path.append(board[i][j-1])
            if self.check_neighbors(i, j - 1, word, path, seen):
                return True
            path.pop(-1)
        # check bottom neighbor
        if i + 1 < len(board) and self.check_candidate_letter(i + 1, j, i, j, seen):
            path.append(board[i+1][j])
            if self.check_neighbors(i + 1, j, word, path, seen):
                return True
            path.pop(-1)
        # check right neighbor
        if j + 1 < len(board[0]) and self.check_candidate_letter(i, j + 1, i, j, seen):
            path.append(board[i][j+1])
            if self.check_neighbors(i, j + 1, word, path, seen):
                return True
            path.pop(-1)
        seen.remove((i, j))  # remove after searching all neighbors
        return False #if no subtree to explore return false
   

class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        # How I'd solve it, Scan through the array and check if there are letters within the word within the board
        # If there are letters within, usually I'd scan neighbors for all candidates and if exists scan again,
        # I'll try with simple greedy for now. 
        # We guarantee that all seen squares we've explored all the possible neighbors with backtracking. 
        # Preprocess valid adjacency neighbors for each character in word
        # Use seen cache to avoid infinite neighbor and back loop of valid 
        helper = Helper(board, word)
        for i in range(len(board)):
            for j in range(len(board[0])):
                if board[i][j] == word[0]:
                    if helper.check_neighbors(i, j, word, [board[i][j]], set()):
                        return True
        return False

In [13]:
def current_solution(board, word):
    return Solution().exist(board, word)

# result = "PASS (No solution provided to execute)"
# print(result)
# When Solution().exist is runnable, replace the two lines above with:
test(current_solution)
print("PASS")


PASS


In [22]:
from typing import List, OrderedDict
import copy

# previous solution was too slow perhaps better seen caching, or check candidate letter 
class Helper:
    def __init__(self, board, word):
        self.board = board
        self.word = word
        # Precache: For each letter in word, record its indexes and its adjacents in word
        # word_resolver[letter] = list of tuples: (index, backward_letter_or_None, forward_letter_or_None)
        self.word_resolver = {}  # letter: list of (idx, backward, forward)
        for idx, c in enumerate(word):
            backward = word[idx - 1] if idx - 1 >= 0 else None
            forward = word[idx + 1] if idx + 1 < len(word) else None
            if c not in self.word_resolver:
                self.word_resolver[c] = []
            self.word_resolver[c].append((idx, backward, forward))

    from collections import deque

    def check_candidate_letter(self, k, l, i, j, seen):
        # Returns list of valid directions for this move: -1 (appendleft), 1 (appendright)
        if (k, l) in seen:
            return []
        letter = self.board[i][j]
        new_letter = self.board[k][l]
        if letter not in self.word_resolver:
            return []
        directions = []
        for idx, backward, forward in self.word_resolver[letter]:
            if backward is not None and backward == new_letter:
                directions.append(-1)
            if forward is not None and forward == new_letter:
                directions.append(1)
        return directions

    def check_neighbors(self, i, j, word, path, seen):
        board = self.board
        if "".join(path) == word:
            return True
        seen.add((i, j))

        directions = [(-1, 0), (0, -1), (1, 0), (0, 1)]  # top, left, bottom, right
        for di, dj in directions:
            ni, nj = i + di, j + dj
            if 0 <= ni < len(board) and 0 <= nj < len(board[0]):
                move_dirs = self.check_candidate_letter(ni, nj, i, j, seen)
                if not move_dirs:
                    continue
                for direction in move_dirs:
                    if direction == -1:
                        path.appendleft(board[ni][nj])
                        found = self.check_neighbors(ni, nj, word, path, seen)
                        path.popleft()
                        if found:
                            seen.remove((i, j))
                            return True
                    elif direction == 1:
                        path.append(board[ni][nj])
                        found = self.check_neighbors(ni, nj, word, path, seen)
                        path.pop()
                        if found:
                            seen.remove((i, j))
                            return True
        seen.remove((i, j))
        return False

 

class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        from collections import deque
        helper = Helper(board, word)
        for i in range(len(board)):
            for j in range(len(board[0])):
                if board[i][j] == word[0]:
                    path = deque([board[i][j]])
                    if helper.check_neighbors(i, j, word, path, set()):
                        return True
        return False
   
 
                
                



In [23]:
def current_solution(board, word):
    return Solution().exist(board, word)

# result = "PASS (No solution provided to execute)"
# print(result)
# When Solution().exist is runnable, replace the two lines above with:
test(current_solution)
print("PASS")


PASS


1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

- Attempt 1 stops at a scan/preprocessing idea, so it never reaches a runnable search strategy.
- Attempt 2 moves to backtracking, which is the right interview direction. Its time complexity is still exponential in the worst case, roughly `O(m * n * 4^L)`, with `L = len(word)`, but that is the standard acceptable shape for this problem because the branching is constrained by local adjacency and visited-state pruning.
- The last attempt adds `word_resolver` and a `deque` so the path can grow on both the left and right. That preprocessing is only `O(L)` space/time, but it does not reduce the real search complexity much; the dominant cost is still recursive exploration.
- The main trade-off in the last attempt is complexity for little payoff: the code becomes harder to reason about, while correctness gets weaker. A concrete failure is `board = [["A", "C", "A"]]`, `word = "AAC"`. The current code returns `True`, even though no valid path spells `A -> A -> C` without illegal re-ordering.
- Why that happens: the final design is not tracking a single forward index into `word`. Instead, it allows the path to be extended around a middle letter using `appendleft` and `append`, which can assemble the target string from both sides rather than following one contiguous traversal order.

2. Critique of the problem-solving approach, including progression of thought and method.

- The progression is healthy: you started by trying to avoid brute-force search, then recognized that adjacency plus no-reuse constraints make this a backtracking problem, not a union-find problem.
- That pivot is the right one. The strong instinct here was noticing that local neighbor validity matters more than global component grouping.
- The weaker part is the next pivot: instead of simplifying to `dfs(i, j, k)` where `k` is the next required character index, the final version tries to be clever with letter-neighbor relationships inside the word. That abstraction drifts away from the actual problem contract.
- Your code is currently validating whether nearby board letters can be adjacent somewhere in the word, not whether the current traversal is matching the exact next position in the word.
- The final solution also carries more state than necessary: `word_resolver`, `deque`, and bidirectional growth all increase reasoning load. For interview settings and production reliability, simpler state with a stronger invariant usually wins.

3. Improvements to Algorithm (Hint-Only Guidance, no full solution code)

- Hint 1: try redefining the recursive state as: current board position plus the exact index `k` you must match next in `word`. Ask yourself what invariant becomes obvious when `k` is explicit.
- Hint 2: in your current code, can a path ever grow "to the left" of the starting character in the target word? Should that be legal for this problem?
- Hint 3: test your implementation on `[["A", "C", "A"]], "AAC"`. Trace `path`, `seen`, and which branch used `appendleft`. Which step creates a string that looks right but was not traversed in legal order?
- Hint 4: if `board[i][j] != word[k]`, should you keep exploring neighbors or fail immediately? That single checkpoint usually removes the need for most of the extra preprocessing.
- Hint 5: once you use an index-based DFS, consider whether you even need to store the full built string/path, or whether only `k` and visited marking are enough.

4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

Transferable systems pattern: constrained path search with local validity checks and no-reuse / no-cycle rules.

Literal usage vs analogy:
- Literal: puzzle solvers, gesture/path validators, grid-based game logic, robot movement over a tiny static map.
- Partial analogy: workflow validation, agent tool-routing constraints, and multi-step state exploration. Those systems usually are not literal 2D grids, but they do share the same design pattern: explore a state graph, prune early, and prevent invalid revisits.

Concrete company/infrastructure examples:
- Big-tech-scale example: a maps or warehouse-routing subsystem may do bounded local search over nearby nodes when validating whether a short maneuver sequence is feasible under immediate adjacency and revisit constraints. The exact LeetCode board model is not used directly, but the pruning discipline is.
- Startup/frontier-tech example: a robotics startup validating whether a manipulator can follow a short pick-path across tightly packed bins can use a small DFS-style feasibility check before invoking a more expensive planner.

2026 AI-agent application mapping:
- Plausible use: an agent orchestration layer can model a short plan as a constrained graph walk where each next tool/action is only legal from certain prior states, and repeated use of the same fragile resource in one plan is disallowed. Backtracking is useful for finding a feasible short action chain.
- Do not use this approach when the agent state space is large, probabilistic, or cost-weighted across many long-horizon branches. In that setting, plain DFS backtracking becomes a poor fit; you would want search with scoring, beam constraints, or learned routing.

Concise application case:
- Context and constraint: an internal support agent must gather account evidence through a fixed set of tools, and some tools cannot be reused in the same audit trail.
- Algorithm/pattern choice: bounded DFS with visited-state control over a small action graph.
- Decision and expected outcome: use DFS/backtracking only for short-horizon validation paths; expect simpler correctness guarantees and early rejection of invalid action sequences.

```mermaid
flowchart TD
    A[Start state] --> B{Next action valid?}
    B -- no --> X[Prune branch]
    B -- yes --> C[Mark state/resource used]
    C --> D{Goal reached?}
    D -- yes --> E[Return feasible path]
    D -- no --> F[Explore adjacent valid states]
    F --> G[Backtrack]
    G --> B
```

When to use this design:
- Small search space, strict local constraints, exact feasibility question, and strong pruning opportunities.

When not to use this design:
- Large global optimization problems, weighted planning, streaming environments, or AI-agent systems where the best next step depends on uncertain future rewards rather than exact local feasibility.

5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

1. In your final code, what invariant guarantees that the `deque` content corresponds to one legal forward walk through `word`, rather than a string assembled around a middle character?
2. Why does `[["A", "C", "A"]], "AAC"` expose a correctness gap? Which exact recursive transition makes the result become a false positive?
3. If you tracked `k`, the current target index in `word`, which parts of `word_resolver` would become unnecessary, and why?
4. Under what board/word patterns does backtracking actually hit its worst-case behavior, and what kinds of early-pruning checks help most before DFS starts?
5. What is the difference between checking whether two letters are adjacent somewhere in the word versus checking whether the next move matches the only legal next index in the current search path?

6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:

- Challenge: Return the path coordinates, not just `True/False`.
  Learning goal intent: practice carrying just enough reconstruction state without breaking the DFS invariant.
  What changed from the original problem: the interface now requires the exact route.
  Why this change matters for design decisions: it tests whether your recursion state is clean and reconstructible.
- Challenge: Allow diagonal movement in addition to up/down/left/right.
  Learning goal intent: understand how branching factor changes complexity and pruning pressure.
  What changed from the original problem: neighborhood size expanded from 4 to 8.
  Why this change matters for design decisions: naive backtracking gets more expensive much faster.
- Challenge: The board is streamed row-by-row, and you only have a small memory budget.
  Learning goal intent: see where this DFS pattern stops fitting and where interface constraints dominate algorithm choice.
  What changed from the original problem: the full board is no longer always resident in memory.
  Why this change matters for design decisions: visited-state and random-access assumptions become much harder to maintain.


In [6]:
from typing import List, OrderedDict
import copy

# previous solution was too slow perhaps better seen caching, or check candidate letter 
class Helper:
    def __init__(self, board, word):
        self.board = board
        self.word = word
        # Precache: For each letter in word, record its indexes and its adjacents in word
        # word_resolver[letter] = list of tuples: (index, backward_letter_or_None, forward_letter_or_None)
        self.word_resolver = {}  # letter: list of (idx, backward, forward)
        for idx, c in enumerate(word):
            forward = word[idx + 1] if idx + 1 < len(word) else None
            if c not in self.word_resolver:
                self.word_resolver[c] = []
            self.word_resolver[c].append((idx, forward))
       


    def check_candidate_letter(self, k, l, i, j, seen):
        # Returns list of valid directions for this move: -1 (appendleft), 1 (appendright)
        if (k, l) in seen:
            return []
        letter = self.board[i][j]
        new_letter = self.board[k][l]
        if letter not in self.word_resolver:
            return []
        directions = []
        for idx, forward in self.word_resolver[letter]:
            if forward is not None and forward == new_letter:
                directions.append(1)
        return directions
   

    def check_neighbors(self, i, j, word, path, seen):
        board = self.board
        if "".join(path) == word:
            return True
        seen.add((i, j))

        directions = [(-1, 0), (0, -1), (1, 0), (0, 1)]  # top, left, bottom, right
        for di, dj in directions:
            ni, nj = i + di, j + dj
            if 0 <= ni < len(board) and 0 <= nj < len(board[0]):
                move_dirs = self.check_candidate_letter(ni, nj, i, j, seen)
                if not move_dirs:
                    continue
                for direction in move_dirs:
                    if direction == 1:
                        path.append(board[ni][nj])
                        found = self.check_neighbors(ni, nj, word, path, seen)
                        path.pop()
                        if found:
                            seen.remove((i, j))
                            return True
                       
        seen.remove((i, j))
        return False

 

class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        helper = Helper(board, word)
        for i in range(len(board)):
            for j in range(len(board[0])):
                if board[i][j] == word[0]:
                    path = [board[i][j]]
                    if helper.check_neighbors(i, j, word, path, set()):
                        return True
        return False
   
 
                
                



In [7]:
def current_solution(board, word):
    return Solution().exist(board, word)

# result = "PASS (No solution provided to execute)"
# print(result)
# When Solution().exist is runnable, replace the two lines above with:
test(current_solution)
print("PASS")


PASS


# Editorial
- Grid traversal problem where a candidate path must match a target string one character at a time.
- The key behaviors are adjacency rules, path reuse restrictions, and early rejection when a partial path already fails.
- Good test coverage should include successful paths, dead ends, repeated letters, and words that nearly match but require reusing a cell.
- Similar patterns show up in puzzle engines, gesture/path validation, board-game move checking, and constrained search in product workflows.
- Focus on correctness around boundaries: corners, single-cell boards, and routes that branch before converging.